**환경 설정 및 라이브러리 설치**

In [1]:
!pip install transformers datasets huggingface_hub

In [1]:
!pip install transformers datasets sentencepiece torch accelerate scikit-learn

import datasets
from datasets import load_dataset, DatasetDict
from transformers import pipeline
import torch
import pandas as pd # 데이터 확인용

device = 0 if torch.cuda.is_available() else -1
print(f"사용 가능한 디바이스: {'GPU' if device == 0 else 'CPU'}")

사용 가능한 디바이스: GPU


**데이터셋 로드 준비 - mteb/amazon_polarity**

In [2]:
!pip install -U datasets

**데이터셋 로드 후 불필요한 데이터 지우기**

In [3]:
from datasets import load_dataset
import pandas as pd

amazon_review_dataset = load_dataset("amazon_polarity",split="train")
print("로드완료\n",amazon_review_dataset)

#일부 데이터만 선택
num_samples_to_use = 30
sample_subset_raw = amazon_review_dataset.select(range(num_samples_to_use))

#불필요한 데이터 지우기
sample_subset = sample_subset_raw.remove_columns(['label'])

print("로드된 데이터셋 정보:")
print(sample_subset)

# 데이터셋 확인을 위해 Pandas DataFrame으로 변환
df_check = pd.DataFrame(sample_subset)
print("\n데이터셋 일부 미리보기 (DataFrame):")
display(df_check.head(10))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

로드완료
 Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 3600000
})
로드된 데이터셋 정보:
Dataset({
    features: ['title', 'content'],
    num_rows: 30
})

데이터셋 일부 미리보기 (DataFrame):


,title,content
0,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...
1,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...
2,Amazing!,This soundtrack is my favorite music of all ti...
3,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...
4,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine..."
5,an absolute masterpiece,I am quite sure any of you actually taking the...
6,Buyer beware,"This is a self-published book, and if you want..."
7,Glorious story,I loved Whisper of the wicked saints. The stor...
8,A FIVE STAR BOOK,I just finished reading Whisper of the Wicked ...
9,Whispers of the Wicked Saints,This was a easy to read book that made me want...


**요약 모델 파이브라인 로딩**

In [4]:
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn",
    device=device
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


**요약하기**

In [5]:
import re

def clean_text(text):
    #2개 이상 반복되는 특수 문자를 공백으로 대체
    text = re.sub(r'[\^\_]{2,}', ' ', text)

    #3번 이상 반복되는 알파벳을 2번으로 줄이기.
    text = re.sub(r'([a-z])\1{2,}', r'\1\1', text, flags=re.IGNORECASE)

    #여러 개의 공백을 하나로 줄이고 앞뒤 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def summarize_content(example):
  cleaned_content =  clean_text(example['content'])
  summary_result = summarizer(
        cleaned_content,
        max_length=150,
        min_length=30,
        do_sample=False
    )
  example['summary'] = summary_result[0]['summary_text']
  return example

summarized_dataset = sample_subset.map(summarize_content)

print("\n요약이 추가된 데이터셋 정보:")
print(summarized_dataset)

# 데이터셋 확인 (Pandas)
df_check_summary = pd.DataFrame(summarized_dataset)
print("\n요약 추가 후 데이터셋 미리보기 (DataFrame):")
display(df_check_summary.head(10))

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Your max_length is set to 150, but your input_length is only 91. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=45)
Your max_length is set to 150, but your input_length is only 110. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=55)
Your max_length is set to 150, but your input_length is only 107. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=53)
Your max_length is set to 150, but your input_length is only 120. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)
Y


요약이 추가된 데이터셋 정보:
Dataset({
    features: ['title', 'content', 'summary'],
    num_rows: 30
})

요약 추가 후 데이터셋 미리보기 (DataFrame):


,title,content,summary
0,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...,Out of all of the games I have ever played it ...
1,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...,The music is timeless and I'm been listening t...
2,Amazing!,This soundtrack is my favorite music of all ti...,This soundtrack is my favorite music of all ti...
3,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...,Xander Cross: I truly like this soundtrack and...
4,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine...","Every single song tells a story of the game, i..."
5,an absolute masterpiece,I am quite sure any of you actually taking the...,This is one of the best videogame soundtracks ...
6,Buyer beware,"This is a self-published book, and if you want...","This is a self-published book, and if you want..."
7,Glorious story,I loved Whisper of the wicked saints. The stor...,The world was raving about this book and so I ...
8,A FIVE STAR BOOK,I just finished reading Whisper of the Wicked ...,"I expected an average romance read, but instea..."
9,Whispers of the Wicked Saints,This was a easy to read book that made me want...,This was a easy to read book that made me want...


**번역 모델 파이브라인 로딩**

In [6]:
translator = pipeline(
    task="translation",
    model="facebook/nllb-200-distilled-600M",
    device=device
)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


**영어를 한국어로 번역하기**

In [7]:
def translate(example):
  translation_result = translator(
      example['summary'],
      src_lang="eng_Latn",
      tgt_lang="kor_Hang",
      max_length=150,
      min_length=30,
      do_sample=False
  )
  example['translation'] = translation_result[0]['translation_text']
  return example

translated_dataset = summarized_dataset.map(translate)

print("\n번역이 추가된 데이터셋 정보:")
print(translated_dataset)

# 데이터셋 확인 (Pandas)
df_check_translation = pd.DataFrame(translated_dataset)
print("\n번역 추가 후 데이터셋 미리보기 (DataFrame):")
display(df_check_translation.head(10))

Map:   0%|          | 0/30 [00:00<?, ? examples/s]


번역이 추가된 데이터셋 정보:
Dataset({
    features: ['title', 'content', 'summary', 'translation'],
    num_rows: 30
})

번역 추가 후 데이터셋 미리보기 (DataFrame):


,title,content,summary,translation
0,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...,Out of all of the games I have ever played it ...,제가 연주한 모든 게임들 중 가장 좋은 음악이 있습니다. 그것은 무작위 키보드 연주...
1,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...,The music is timeless and I'm been listening t...,"음악은 시대적이고, 저는 몇 년 동안 그것을 듣고 있습니다. 그리고 그 아름다움은 ..."
2,Amazing!,This soundtrack is my favorite music of all ti...,This soundtrack is my favorite music of all ti...,"이 사운드트랙은 모든 시간의 가장 좋아하는 음악입니다. ""재정의 포로들""의 강렬한 ..."
3,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...,Xander Cross: I truly like this soundtrack and...,젠더 크로스: 이 사운드트랙은 정말 좋아요. 비디오 게임 음악도 좋아해요. 저는 이...
4,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine...","Every single song tells a story of the game, i...",모든 노래는 게임에 대한 이야기를 들려줍니다. 그것은 그렇게 좋습니다! 가장 위대한...
5,an absolute masterpiece,I am quite sure any of you actually taking the...,This is one of the best videogame soundtracks ...,이것은 최고의 비디오 게임 사운드트랙 중 하나입니다. 그리고 확실히 미쓰다의 최고입...
6,Buyer beware,"This is a self-published book, and if you want...","This is a self-published book, and if you want...","이건 자기 출판된 책이고, 그 이유를 알고 싶다면 몇 장을 읽어봐요. 5성평가는 허..."
7,Glorious story,I loved Whisper of the wicked saints. The stor...,The world was raving about this book and so I ...,전 세계가 이 책에 대해 화제를 모았기 때문에 나는 그것을 샀다. 나는 그것을 사랑...
8,A FIVE STAR BOOK,I just finished reading Whisper of the Wicked ...,"I expected an average romance read, but instea...","저는 평균적인 낭만적 인 책을 읽기를 기대했지만, 그 대신 제가 가장 좋아하는 책 ..."
9,Whispers of the Wicked Saints,This was a easy to read book that made me want...,This was a easy to read book that made me want...,"이 책은 읽기 쉬운 책이었는데, 계속 읽으려는 마음이 들었고, 놓으려는 마음이 들지..."


**분류 모델 파이프라인 로딩**

In [8]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


**카테고리 분류하기**

In [10]:
def classification(example):
    candidate_labels = [
    "Beauty",
    "Fashion",
    "Food",
    "Beverages",
    "Home Appliances",
    "Electronics",
    "Books",
    "Health",
    "Toys",
    "Sports",
    "Kitchenware",
    "Music",
    "Gaming"
    ]

    classification_result = classifier(
        example['summary'],
        candidate_labels=candidate_labels
    )

    # 가장 높은 점수의 카테고리를 저장
    example['category'] = classification_result['labels'][0]

    return example

classified_dataset = translated_dataset.map(classification)
print(classified_dataset)

df_check_classify = pd.DataFrame(classified_dataset)
print("\n번역 추가 후 데이터셋 미리보기 (DataFrame):")
display(df_check_classify.head(10))

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'content', 'summary', 'translation', 'category'],
    num_rows: 30
})

번역 추가 후 데이터셋 미리보기 (DataFrame):


,title,content,summary,translation,category
0,Stuning even for the non-gamer,This sound track was beautiful! It paints the ...,Out of all of the games I have ever played it ...,제가 연주한 모든 게임들 중 가장 좋은 음악이 있습니다. 그것은 무작위 키보드 연주...,Music
1,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...,The music is timeless and I'm been listening t...,"음악은 시대적이고, 저는 몇 년 동안 그것을 듣고 있습니다. 그리고 그 아름다움은 ...",Music
2,Amazing!,This soundtrack is my favorite music of all ti...,This soundtrack is my favorite music of all ti...,"이 사운드트랙은 모든 시간의 가장 좋아하는 음악입니다. ""재정의 포로들""의 강렬한 ...",Music
3,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...,Xander Cross: I truly like this soundtrack and...,젠더 크로스: 이 사운드트랙은 정말 좋아요. 비디오 게임 음악도 좋아해요. 저는 이...,Music
4,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine...","Every single song tells a story of the game, i...",모든 노래는 게임에 대한 이야기를 들려줍니다. 그것은 그렇게 좋습니다! 가장 위대한...,Music
5,an absolute masterpiece,I am quite sure any of you actually taking the...,This is one of the best videogame soundtracks ...,이것은 최고의 비디오 게임 사운드트랙 중 하나입니다. 그리고 확실히 미쓰다의 최고입...,Gaming
6,Buyer beware,"This is a self-published book, and if you want...","This is a self-published book, and if you want...","이건 자기 출판된 책이고, 그 이유를 알고 싶다면 몇 장을 읽어봐요. 5성평가는 허...",Books
7,Glorious story,I loved Whisper of the wicked saints. The stor...,The world was raving about this book and so I ...,전 세계가 이 책에 대해 화제를 모았기 때문에 나는 그것을 샀다. 나는 그것을 사랑...,Books
8,A FIVE STAR BOOK,I just finished reading Whisper of the Wicked ...,"I expected an average romance read, but instea...","저는 평균적인 낭만적 인 책을 읽기를 기대했지만, 그 대신 제가 가장 좋아하는 책 ...",Books
9,Whispers of the Wicked Saints,This was a easy to read book that made me want...,This was a easy to read book that made me want...,"이 책은 읽기 쉬운 책이었는데, 계속 읽으려는 마음이 들었고, 놓으려는 마음이 들지...",Books
